<a href="https://colab.research.google.com/github/hectorn61/Mis_archivos/blob/master/Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from datetime import datetime
import docx
from docx.shared import Inches
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
import re
from scipy.stats import chi2_contingency
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.cluster.hierarchy import linkage, dendrogram

def add_df_to_word(doc, df, heading):
    """Adds a Pandas DataFrame to a Word document."""
    doc.add_heading(heading, level=2)
    table = doc.add_table(rows=df.shape[0] + 1, cols=df.shape[1])

    # Header row
    for j in range(df.shape[1]):
        table.cell(0, j).text = df.columns[j]

    # Data rows
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            table.cell(i + 1, j).text = str(df.iloc[i, j])

def sanitize_filename(filename):
    """Sanitizes the filename by replacing invalid characters with '_'."""
    return re.sub(r'[<>:"/\\|?*]', "_", filename)

def plot_scatterplots(data_numeric, timestamp, folder_path):
    """Generates scatterplots of all possible pairs of variables in data_numeric."""
    numeric_cols = data_numeric.columns
    n_cols = len(numeric_cols)

    # Create a specific folder for the scatterplots
    scatterplots_folder = os.path.join(folder_path, "scatterplots")
    os.makedirs(scatterplots_folder, exist_ok=True)

    # Generate scatterplots for all pairs of variables
    for i, j in combinations(range(n_cols), 2):
        col1 = numeric_cols[i]
        col2 = numeric_cols[j]

        # Sanitize the column names
        col1_sanitized = sanitize_filename(col1)
        col2_sanitized = sanitize_filename(col2)

        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=data_numeric[col1], y=data_numeric[col2], alpha=0.6)
        plt.title(f"Scatterplot: {col1} vs {col2}")
        plt.xlabel(col1)
        plt.ylabel(col2)

        # Save the plot
        filename = os.path.join(
            scatterplots_folder,
            f"scatterplot_{col1_sanitized}_vs_{col2_sanitized}_{timestamp}.png",
        )
        plt.savefig(filename)
        plt.close()
        print(f"Scatterplot de {col1} vs {col2} guardado en '{filename}'.")
    return scatterplots_folder

def plot_dendrogram(data_numeric, timestamp, folder_path):
    """Generates a dendrogram from the numeric data."""
    # Calculate the linkage matrix
    Z = linkage(data_numeric, method="ward")

    # Create the dendrogram
    plt.figure(figsize=(12, 8))
    dendrogram(Z, labels=data_numeric.index, leaf_rotation=90)
    plt.title("Dendrograma de Clusters Jerárquicos")
    plt.xlabel("Índice de la Observación")
    plt.ylabel("Distancia")

    # Save the plot
    filename = os.path.join(folder_path, f"dendrogram_{timestamp}.png")
    plt.savefig(filename)
    plt.close()
    print(f"Dendrograma guardado en '{filename}'.")
    return filename

def plot_dendrogram_variables(data_numeric, timestamp, folder_path, figsize=(15, 8), font_size=10):
    """Generates a dendrogram showing the relationships between the variables."""
    # Calculate the correlation matrix
    corr_matrix = data_numeric.corr()

    # Calculate the linkage matrix
    Z = linkage(corr_matrix, method="ward")

    # Create the dendrogram
    plt.figure(figsize=figsize)
    dendrogram(Z, labels=corr_matrix.columns, leaf_rotation=90, leaf_font_size=font_size)
    plt.title("Dendrograma de Relaciones entre Variables", fontsize=16)
    plt.xlabel("Variables", fontsize=16)
    plt.ylabel("Distancia (Correlación)", fontsize=16)

    # Save the plot
    filename = os.path.join(folder_path, f"dendrogram_variables_{timestamp}.png")
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    print(f"Dendrograma de variables guardado en '{filename}'.")
    return filename

def plot_correlation_circle_improved(loadings, features, explained_var, title, timestamp, folder_path, rotation_type=""):
    """Generates an improved correlation circle plot with variance percentages."""
    plt.figure(figsize=(10, 10))
    ax = plt.gca()

    # Correlation circle
    circle = plt.Circle((0, 0), 1, color='blue', fill=False)
    ax.add_artist(circle)

    # Plot limits
    plt.xlim(-1.1, 1.1)
    plt.ylim(-1.1, 1.1)

    # Axes with explained variance
    plt.xlabel(f"Componente 1 ({explained_var[0]*100:.1f}%)", fontsize=12)
    plt.ylabel(f"Componente 2 ({explained_var[1]*100:.1f}%)", fontsize=12)

    # Plot each variable
    for i, feature in enumerate(features):
        x = loadings[i, 0]
        y = loadings[i, 1]
        plt.arrow(0, 0, x, y, head_width=0.03, head_length=0.05, fc='red', ec='red')
        plt.text(x * 1.15, y * 1.15, feature, color='black', ha='center', va='center', fontsize=9)

    # Grid and reference lines
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(0, color='black', linestyle='--', linewidth=0.5)
    plt.title(title, fontsize=14, pad=20)

    # Save the plot
    filename = os.path.join(folder_path, f"correlation_circle_{rotation_type}_{timestamp}.png")
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    return filename

def plot_variance_explained(variance, title, timestamp, folder_path, rotation_type=""):
    """Plots individual and cumulative explained variance."""
    cumulative_variance = np.cumsum(variance)

    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(1, len(variance)+1), variance, alpha=0.6, color='steelblue', label='Varianza Individual')
    plt.plot(range(1, len(variance)+1), cumulative_variance, 'ro-', label='Varianza Acumulada')

    # Add values to bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                 f'{height*100:.1f}%',
                 ha='center', va='bottom', fontsize=9)

    # Add values to cumulative line
    for i, v in enumerate(cumulative_variance):
        plt.text(i+1, v+0.02, f'{v*100:.1f}%', ha='center', va='bottom', color='red')

    plt.xlabel('Número de Componentes', fontsize=12)
    plt.ylabel('Varianza Explicada', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(range(1, len(variance)+1))

    filename = os.path.join(folder_path, f"variance_explained_{rotation_type}_{timestamp}.png")
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    return filename

def plot_correlation_matrix(corr_matrix, timestamp, folder_path, figsize=(15, 12), font_scale=0.9):
    """Visualizes the correlation matrix with custom size and font adjustment."""
    # Adjust the font size
    sns.set(font_scale=font_scale)
    # Create the plot
    plt.figure(figsize=figsize)
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("Matriz de Correlación", fontsize=16)

    # Save the plot
    filename = os.path.join(folder_path, f"correlation_matrix_{timestamp}.png")
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    print(f"Matriz de correlación guardada en '{filename}'.")
    return filename

def plot_pca_clusters(pca_scores, cluster_labels, timestamp, folder_path):
    """Generates scatterplot of PCA1 vs PCA2 with clusters."""
    plt.figure(figsize=(12, 8))
    scatter = sns.scatterplot(
        x=pca_scores[:, 0],
        y=pca_scores[:, 1],
        hue=cluster_labels,
        palette="viridis",
        s=100,
        alpha=0.8
    )

    plt.title("PCA: Componente 1 vs Componente 2 con Clusters", fontsize=16)
    plt.xlabel("Componente Principal 1", fontsize=12)
    plt.ylabel("Componente Principal 2", fontsize=12)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')

    # Añadir centroides
    centroids = []
    for cluster in np.unique(cluster_labels):
        centroids.append(pca_scores[cluster_labels == cluster].mean(axis=0))
    centroids = np.array(centroids)

    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='X', s=200, c='red',
        label='Centroides', edgecolor='black'
    )

    # Save the plot
    filename = os.path.join(folder_path, f"pca_clusters_{timestamp}.png")
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    return filename

def determine_optimal_clusters(data, max_k=7):
    """Determines the optimal number of clusters using the elbow method and silhouette score."""
    inertias = []
    silhouette_scores = []
    k_values = range(2, max_k+1)

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        clusters = kmeans.fit_predict(data)
        inertias.append(kmeans.inertia_)

        if len(set(clusters)) > 1:  # Silhouette needs at least 2 clusters
            silhouette_scores.append(silhouette_score(data, clusters))
        else:
            silhouette_scores.append(0)

    # Basic elbow method (can be enhanced)
    optimal_k = k_values[np.argmin(inertias)]

    return optimal_k

def plot_biplot(scores_pc1, scores_pc2, loadings_pc1, loadings_pc2, features, timestamp, folder_path):
    """Generates a biplot from PCA scores and loadings."""
    plt.figure(figsize=(12, 8))

    # Plot the PCA scores (samples)
    plt.scatter(scores_pc1, scores_pc2, marker='o', alpha=0.5, label="Muestras")

    # Plot the loadings (variables)
    scale_arrow = max(np.abs(scores_pc1).max(), np.abs(scores_pc2).max()) / max(np.abs(loadings_pc1).max(), np.abs(loadings_pc2).max())
    for i, feature in enumerate(features):
        plt.arrow(0, 0, loadings_pc1[i] * scale_arrow, loadings_pc2[i] * scale_arrow,
                  color='r', alpha=0.7, head_width=0.05 * scale_arrow, head_length=0.05 * scale_arrow)
        plt.text(loadings_pc1[i] * scale_arrow * 1.1, loadings_pc2[i] * scale_arrow * 1.1,
                 feature, color='g', ha='center', va='center')

    plt.xlabel("Componente Principal 1")
    plt.ylabel("Componente Principal 2")
    plt.title("Biplot de PCA (Componentes 1 y 2)")
    plt.grid(True)
    plt.legend()

    # Save the plot
    filename = os.path.join(folder_path, f"biplot_{timestamp}.png")
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    print(f"Biplot guardado en '{filename}'.")
    return filename

def perform_anova(data, dependent_variable, grouping_variable):
    """Performs ANOVA and returns the results."""
    formula = f'{dependent_variable} ~ C({grouping_variable})'
    model = ols(formula, data=data).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    return anova_table

def perform_chi2(data, col1, col2):
    """Performs Chi-Square test of independence and returns the results."""
    contingency_table = pd.crosstab(data[col1], data[col2])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    return chi2, p, dof, expected

def comprehensive_pca_analysis(data_numeric, timestamp, folder_path, doc):
    """Performs complete PCA analysis with rotation and evaluation."""

    # Separar columnas numéricas y de texto
    data_numeric = data.select_dtypes(include=[np.number])
    data_text = data.select_dtypes(exclude=[np.number])

    # Scale data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data_numeric)

    # KMO and Bartlett's Test
    kmo_all, kmo_model = calculate_kmo(data_numeric)
    bartlett_chi2, bartlett_p = calculate_bartlett_sphericity(data_numeric)

    # Adequacy evaluation
    kmo_evaluation = "Excelente" if kmo_model >= 0.9 else \
                     "Muy bueno" if kmo_model >= 0.8 else \
                     "Aceptable" if kmo_model >= 0.7 else \
                     "Mediocre" if kmo_model >= 0.6 else \
                     "Inaceptable"

    bartlett_evaluation = "Adecuado (p < 0.05)" if bartlett_p < 0.05 else "No adecuado (p >= 0.05)"

    # PCA without rotation
    pca_unrotated = PCA(n_components=min(data_scaled.shape[1], data_scaled.shape[0]))
    pca_unrotated.fit(data_scaled)
    pca_unrotated_scores = pca_unrotated.transform(data_scaled)

    # PCA with Varimax rotation
    fa = FactorAnalyzer(rotation='varimax', n_factors=min(data_scaled.shape[1], data_scaled.shape[0]))
    fa.fit(data_scaled)
    rotated_loadings = fa.loadings_

    # Unrotated PCA results
    unrotated_loadings = pca_unrotated.components_.T * np.sqrt(pca_unrotated.explained_variance_)
    unrotated_loadings_df = pd.DataFrame(
        unrotated_loadings,
        columns=[f"PC{i+1}" for i in range(pca_unrotated.n_components_)],
        index=data_numeric.columns
    )

    # Rotated PCA results
    rotated_loadings_df = pd.DataFrame(
        rotated_loadings,
        columns=[f"Factor{i+1}" for i in range(fa.n_factors)],
        index=data_numeric.columns
    )

    # Clustering
    optimal_k = determine_optimal_clusters(data_scaled)
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(data_scaled)

    # Create DataFrames for Excel output
    pca_scores_df = pd.DataFrame(pca_unrotated_scores, columns=[f"PC{i+1}" for i in range(pca_unrotated.n_components_)])
    pca_scores_df['Cluster'] = cluster_labels  # Add cluster assignments
    pca_scores_df['Sample'] = data_numeric.index  # Add sample names (if available)

    # Create a new dataframe including the original data, PCA scores, and cluster
    data_with_clusters = data_numeric.copy()
    data_with_clusters['Cluster'] = cluster_labels

    # Explained variance without rotation
    variance_unrotated = pd.DataFrame({
        "Componente": [f"PC{i+1}" for i in range(pca_unrotated.n_components_)],
        "Varianza Individual": pca_unrotated.explained_variance_ratio_,
        "Varianza Acumulada": np.cumsum(pca_unrotated.explained_variance_ratio_)
    })

    # Explained variance with rotation
    variance_rotated = pd.DataFrame({
        "Factor": [f"Factor{i+1}" for i in range(fa.n_factors)],
        "Varianza Individual": fa.get_factor_variance()[1],
        "Varianza Acumulada": np.cumsum(fa.get_factor_variance()[1])
    })

    # Generate plots
    var_unrotated_plot = plot_variance_explained(pca_unrotated.explained_variance_ratio_, "Varianza Explicada (PCA sin rotación)", timestamp, folder_path, "unrotated")
    var_rotated_plot = plot_variance_explained(fa.get_factor_variance()[1], "Varianza Explicada (PCA con rotación Varimax)", timestamp, folder_path, "rotated")
    circle_unrotated = plot_correlation_circle_improved(unrotated_loadings[:, :2], data_numeric.columns, pca_unrotated.explained_variance_ratio_[:2], "Círculo de Correlación (PCA sin rotación - Componentes 1-2)", timestamp, folder_path, "unrotated")
    circle_rotated = plot_correlation_circle_improved(rotated_loadings[:, :2], data_numeric.columns, fa.get_factor_variance()[1][:2], "Círculo de Correlación (PCA con rotación Varimax)", timestamp, folder_path, "rotated")
    biplot_filename = plot_biplot(pca_unrotated_scores[:, 0], pca_unrotated_scores[:, 1], unrotated_loadings[:, 0], unrotated_loadings[:, 1], data_numeric.columns, timestamp, folder_path)
    pca = PCA(n_components=2)
    pca_scores = pca.fit_transform(data_scaled)
    clusters_plot = plot_pca_clusters(pca_scores, cluster_labels, timestamp, folder_path)

    # Excel output
    excel_filename = os.path.join(folder_path, f"PCA_Results_{timestamp}.xlsx")
    with pd.ExcelWriter(excel_filename) as writer:
        pca_scores_df.to_excel(writer, sheet_name="PCA Scores and Clusters", index=False)
        unrotated_loadings_df.to_excel(writer, sheet_name="Unrotated Loadings", index=True)
        rotated_loadings_df.to_excel(writer, sheet_name="Rotated Loadings", index=True)
        data_with_clusters.to_excel(writer, sheet_name="Data With Clusters", index=True)

        # Calculate and add cluster sizes
        cluster_sizes = pd.DataFrame(data_with_clusters['Cluster'].value_counts())
        cluster_sizes.to_excel(writer, sheet_name="Cluster Sizes")

        # Calculate and add cluster means
        cluster_means = data_with_clusters.groupby('Cluster').mean()
        cluster_means.to_excel(writer, sheet_name="Cluster Means")

    # Word output
    doc.add_heading("Análisis de Componentes Principales (PCA)", level=1)
    doc.add_heading("Evaluación de Adecuación del Modelo", level=2)
    doc.add_paragraph(f"KMO del modelo: {kmo_model:.3f} ({kmo_evaluation})")
    doc.add_paragraph(f"Prueba de Bartlett: chi2={bartlett_chi2:.3f}, p={bartlett_p:.3f} ({bartlett_evaluation})")

    add_df_to_word(doc, variance_unrotated, "Varianza Explicada (Sin Rotación)")
    doc.add_picture(var_unrotated_plot, width=Inches(6))
    # doc.add_df_to_word(doc, variance_rotated, "Varianza Explicada (Con Rotación Varimax)") #This is line 414
    doc.add_picture(var_rotated_plot, width=Inches(6))
    doc.add_picture(circle_unrotated, width=Inches(6))
    doc.add_picture(circle_rotated, width=Inches(6))
    doc.add_picture(biplot_filename, width=Inches(6))

    if clusters_plot:
        doc.add_heading("Visualización de Clusters", level=2)
        doc.add_picture(clusters_plot, width=Inches(6))

    # Identify the variables most associated with each cluster

    doc.add_heading("Variables Relevantes por Cluster", level=2)

    cluster_means = data_with_clusters.groupby('Cluster').mean()

    for cluster in cluster_means.index:

        cluster_desc = cluster_means.loc[cluster].sort_values(ascending=False)

        doc.add_paragraph(f"Cluster {cluster}:")

        for var, value in cluster_desc.items():

            doc.add_paragraph(f"\t{var}: {value:.2f}")

    return data_with_clusters

def analyze_data(df, folder_path, timestamp, doc):
    """Analyzes the data and saves results to a Word document."""
    doc.add_heading("Análisis Exploratorio de Datos", level=1)

    # Descriptive statistics
    doc.add_heading("Estadísticas Descriptivas", level=2)
    desc_stats = df.describe()
    add_df_to_word(doc, desc_stats, "Estadísticas Descriptivas")

    # Correlation matrix
    doc.add_heading("Matriz de Correlación", level=2)
    corr_matrix = df.corr(numeric_only=True)
    corr_matrix_plot = plot_correlation_matrix(corr_matrix, timestamp, folder_path)
    doc.add_picture(corr_matrix_plot, width=Inches(6))

    # Scatter plots
    doc.add_heading("Gráficos de Dispersión", level=2)
    numeric_df = df.select_dtypes(include=np.number)
    if len(numeric_df.columns) > 1:
        scatter_plots_folder = plot_scatterplots(numeric_df, timestamp, folder_path)
        # You might want to add some scatterplot images to the Word document
    else:
        doc.add_paragraph("No hay suficientes columnas numéricas para generar gráficos de dispersión.")

    # Dendrogram
    doc.add_heading("Dendrogramas", level=2)
    if len(numeric_df.columns) > 1:
        dendrogram_filename = plot_dendrogram_variables(numeric_df, timestamp, folder_path)
        doc.add_picture(dendrogram_filename, width=Inches(6))
    else:
        doc.add_paragraph("No hay suficientes columnas numéricas para generar dendrogramas.")

    # ANOVA example
    if df.select_dtypes(include='number').shape[1] > 0 and df.select_dtypes(exclude='number').shape[1] > 0:
        try:
            numeric_col = df.select_dtypes(include='number').columns[0]
            categorical_col = df.select_dtypes(exclude='number').columns[0]
            anova_results = perform_anova(df, numeric_col, categorical_col)
            doc.add_heading("Análisis de Varianza (ANOVA)", level=2)
            add_df_to_word(doc, anova_results, f"ANOVA: {numeric_col} vs {categorical_col}")
        except Exception as e:
            doc.add_paragraph(f"No se pudo realizar ANOVA: {e}")

    # Chi-Square test example
    categorical_cols = df.select_dtypes(exclude='number').columns
    if len(categorical_cols) >= 2:
        try:
            chi2, p, dof, expected = perform_chi2(df, categorical_cols[0], categorical_cols[1])
            doc.add_heading("Prueba de Independencia Chi-Cuadrado", level=2)
            doc.add_paragraph(f"Chi2 = {chi2:.3f}, p = {p:.3f}, Grados de libertad = {dof}")
            # You could also add the expected values to the document if needed
        except Exception as e:
            doc.add_paragraph(f"No se pudo realizar la prueba Chi-Cuadrado: {e}")

    # PCA
    numeric_df = df.select_dtypes(include=np.number)
    if len(numeric_df.columns) > 1:
        comprehensive_pca_analysis(numeric_df, timestamp, folder_path, doc)
    else:
        doc.add_paragraph("No hay suficientes columnas numéricas para realizar PCA.")

def main():
    # Load the Excel file
    try:
        df = pd.read_excel("Data.xlsx")
    except FileNotFoundError:
        print("Error: El archivo 'Data.xlsx' no se encuentra. Asegúrate de que esté en el mismo directorio que este script.")
        return

    # Create timestamped folder
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    folder_path = os.path.join(os.getcwd(), f"Analisis_{timestamp}")
    os.makedirs(folder_path, exist_ok=True)
    print(f"Carpeta de resultados creada en: {folder_path}")

    # Create a Word document
    doc = docx.Document()

    # Perform data analysis
    analyze_data(df, folder_path, timestamp, doc)

    # Save the Word document
    doc.save(os.path.join(folder_path, f"Analisis_{timestamp}.docx"))
    print(f"Análisis completado. Los resultados se han guardado en 'Analisis_{timestamp}.docx' y 'PCA_Results_{timestamp}.xlsx' dentro de la carpeta '{folder_path}'.")

if __name__ == "__main__":
    main()

Carpeta de resultados creada en: /content/Analisis_20250411_135722
Matriz de correlación guardada en '/content/Analisis_20250411_135722/correlation_matrix_20250411_135722.png'.
Scatterplot de g/t Pb vs g/t Zn guardado en '/content/Analisis_20250411_135722/scatterplots/scatterplot_g_t Pb_vs_g_t Zn_20250411_135722.png'.
Scatterplot de g/t Pb vs g/t Fe guardado en '/content/Analisis_20250411_135722/scatterplots/scatterplot_g_t Pb_vs_g_t Fe_20250411_135722.png'.
Scatterplot de g/t Pb vs g/t Ag guardado en '/content/Analisis_20250411_135722/scatterplots/scatterplot_g_t Pb_vs_g_t Ag_20250411_135722.png'.
Scatterplot de g/t Pb vs g/t Cu guardado en '/content/Analisis_20250411_135722/scatterplots/scatterplot_g_t Pb_vs_g_t Cu_20250411_135722.png'.
Scatterplot de g/t Pb vs % C total guardado en '/content/Analisis_20250411_135722/scatterplots/scatterplot_g_t Pb_vs_% C total_20250411_135722.png'.
Scatterplot de g/t Pb vs % S total guardado en '/content/Analisis_20250411_135722/scatterplots/scatter

/usr/local/lib/python3.11/dist-packages/factor_analyzer/utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Biplot guardado en '/content/Analisis_20250411_135722/biplot_20250411_135722.png'.
Análisis completado. Los resultados se han guardado en 'Analisis_20250411_135722.docx' y 'PCA_Results_20250411_135722.xlsx' dentro de la carpeta '/content/Analisis_20250411_135722'.


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from datetime import datetime
import docx
from docx.shared import Inches
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
import re
from scipy.stats import chi2_contingency
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.cluster.hierarchy import linkage, dendrogram

def add_df_to_word(doc, df, heading):
    """Adds a Pandas DataFrame to a Word document."""
    doc.add_heading(heading, level=2)
    table = doc.add_table(rows=df.shape[0] + 1, cols=df.shape[1])

    # Header row
    for j in range(df.shape[1]):
        table.cell(0, j).text = df.columns[j]

    # Data rows
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            table.cell(i + 1, j).text = str(df.iloc[i, j])

def sanitize_filename(filename):
    """Sanitizes the filename by replacing invalid characters with '_'."""
    return re.sub(r'[<>:"/\\|?*]', "_", filename)

def plot_scatterplots(data_numeric, timestamp, folder_path):
    """Generates scatterplots of all possible pairs of variables in data_numeric."""
    numeric_cols = data_numeric.columns
    n_cols = len(numeric_cols)

    # Create a specific folder for the scatterplots
    scatterplots_folder = os.path.join(folder_path, "scatterplots")
    os.makedirs(scatterplots_folder, exist_ok=True)

    # Generate scatterplots for all pairs of variables
    for i, j in combinations(range(n_cols), 2):
        col1 = numeric_cols[i]
        col2 = numeric_cols[j]

        # Sanitize the column names
        col1_sanitized = sanitize_filename(col1)
        col2_sanitized = sanitize_filename(col2)

        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=data_numeric[col1], y=data_numeric[col2], alpha=0.6)
        plt.title(f"Scatterplot: {col1} vs {col2}")
        plt.xlabel(col1)
        plt.ylabel(col2)

        # Save the plot
        filename = os.path.join(
            scatterplots_folder,
            f"scatterplot_{col1_sanitized}_vs_{col2_sanitized}_{timestamp}.png",
        )
        plt.savefig(filename)
        plt.close()
        print(f"Scatterplot de {col1} vs {col2} guardado en '{filename}'.")
    return scatterplots_folder

def plot_dendrogram(data_numeric, timestamp, folder_path):
    """Generates a dendrogram from the numeric data."""
    # Calculate the linkage matrix
    Z = linkage(data_numeric, method="ward")

    # Create the dendrogram
    plt.figure(figsize=(12, 8))
    dendrogram(Z, labels=data_numeric.index, leaf_rotation=90)
    plt.title("Dendrograma de Clusters Jerárquicos")
    plt.xlabel("Índice de la Observación")
    plt.ylabel("Distancia")

    # Save the plot
    filename = os.path.join(folder_path, f"dendrogram_{timestamp}.png")
    plt.savefig(filename)
    plt.close()
    print(f"Dendrograma guardado en '{filename}'.")
    return filename

def plot_dendrogram_variables(data_numeric, timestamp, folder_path, figsize=(15, 8), font_size=10):
    """Generates a dendrogram showing the relationships between the variables."""
    # Calculate the correlation matrix
    corr_matrix = data_numeric.corr()

    # Calculate the linkage matrix
    Z = linkage(corr_matrix, method="ward")

    # Create the dendrogram
    plt.figure(figsize=figsize)
    dendrogram(Z, labels=corr_matrix.columns, leaf_rotation=90, leaf_font_size=font_size)
    plt.title("Dendrograma de Relaciones entre Variables", fontsize=16)
    plt.xlabel("Variables", fontsize=16)
    plt.ylabel("Distancia (Correlación)", fontsize=16)

    # Save the plot
    filename = os.path.join(folder_path, f"dendrogram_variables_{timestamp}.png")
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    print(f"Dendrograma de variables guardado en '{filename}'.")
    return filename

def plot_correlation_circle_improved(loadings, features, explained_var, title, timestamp, folder_path, rotation_type=""):
    """Generates an improved correlation circle plot with variance percentages."""
    plt.figure(figsize=(10, 10))
    ax = plt.gca()

    # Correlation circle
    circle = plt.Circle((0, 0), 1, color='blue', fill=False)
    ax.add_artist(circle)

    # Plot limits
    plt.xlim(-1.1, 1.1)
    plt.ylim(-1.1, 1.1)

    # Axes with explained variance
    plt.xlabel(f"Componente 1 ({explained_var[0]*100:.1f}%)", fontsize=12)
    plt.ylabel(f"Componente 2 ({explained_var[1]*100:.1f}%)", fontsize=12)

    # Plot each variable
    for i, feature in enumerate(features):
        x = loadings[i, 0]
        y = loadings[i, 1]
        plt.arrow(0, 0, x, y, head_width=0.03, head_length=0.05, fc='red', ec='red')
        plt.text(x * 1.15, y * 1.15, feature, color='black', ha='center', va='center', fontsize=9)

    # Grid and reference lines
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(0, color='black', linestyle='--', linewidth=0.5)
    plt.title(title, fontsize=14, pad=20)

    # Save the plot
    filename = os.path.join(folder_path, f"correlation_circle_{rotation_type}_{timestamp}.png")
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    return filename

def plot_variance_explained(variance, title, timestamp, folder_path, rotation_type=""):
    """Plots individual and cumulative explained variance."""
    cumulative_variance = np.cumsum(variance)

    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(1, len(variance)+1), variance, alpha=0.6, color='steelblue', label='Varianza Individual')
    plt.plot(range(1, len(variance)+1), cumulative_variance, 'ro-', label='Varianza Acumulada')

    # Add values to bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                 f'{height*100:.1f}%',
                 ha='center', va='bottom', fontsize=9)

    # Add values to cumulative line
    for i, v in enumerate(cumulative_variance):
        plt.text(i+1, v+0.02, f'{v*100:.1f}%', ha='center', va='bottom', color='red')

    plt.xlabel('Número de Componentes', fontsize=12)
    plt.ylabel('Varianza Explicada', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.xticks(range(1, len(variance)+1))

    filename = os.path.join(folder_path, f"variance_explained_{rotation_type}_{timestamp}.png")
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    return filename

def plot_correlation_matrix(corr_matrix, timestamp, folder_path, figsize=(15, 12), font_scale=0.9):
    """Visualizes the correlation matrix with custom size and font adjustment."""
    # Adjust the font size
    sns.set(font_scale=font_scale)
    # Create the plot
    plt.figure(figsize=figsize)
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("Matriz de Correlación", fontsize=16)

    # Save the plot
    filename = os.path.join(folder_path, f"correlation_matrix_{timestamp}.png")
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    print(f"Matriz de correlación guardada en '{filename}'.")
    return filename

def plot_pca_clusters(pca_scores, cluster_labels, timestamp, folder_path):
    """Generates scatterplot of PCA1 vs PCA2 with clusters."""
    plt.figure(figsize=(12, 8))
    scatter = sns.scatterplot(
        x=pca_scores[:, 0],
        y=pca_scores[:, 1],
        hue=cluster_labels,
        palette="viridis",
        s=100,
        alpha=0.8
    )
    plt.title("PCA: Componente 1 vs Componente 2 con Clusters", fontsize=16)
    plt.xlabel("Componente Principal 1", fontsize=12)
    plt.ylabel("Componente Principal 2", fontsize=12)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')

    # Añadir centroides
    centroids = []
    for cluster in np.unique(cluster_labels):
        centroids.append(pca_scores[cluster_labels == cluster].mean(axis=0))
    centroids = np.array(centroids)

    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='X', s=200, c='red',
        label='Centroides', edgecolor='black'
    )

    # Save the plot
    filename = os.path.join(folder_path, f"pca_clusters_{timestamp}.png")
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close()
    return filename

def determine_optimal_clusters(data, max_k=7):
    """Determines the optimal number of clusters using the elbow method and silhouette score."""
    inertias = []
    silhouette_scores = []
    k_values = range(2, max_k+1)

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        clusters = kmeans.fit_predict(data)
        inertias.append(kmeans.inertia_)

        if len(set(clusters)) > 1:  # Silhouette needs at least 2 clusters
            silhouette_scores.append(silhouette_score(data, clusters))
        else:
            silhouette_scores.append(0)

    # Basic elbow method (can be enhanced)
    optimal_k = k_values[np.argmin(inertias)]

    return optimal_k

def plot_biplot(scores_pc1, scores_pc2, loadings_pc1, loadings_pc2, features, timestamp, folder_path):
    """Generates a biplot from PCA scores and loadings."""
    plt.figure(figsize=(12, 8))

    # Plot the PCA scores (samples)
    plt.scatter(scores_pc1, scores_pc2, marker='o', alpha=0.5, label="Muestras")

    # Plot the loadings (variables)
    scale_arrow = max(np.abs(scores_pc1).max(), np.abs(scores_pc2).max()) / max(np.abs(loadings_pc1).max(), np.abs(loadings_pc2).max())
    for i, feature in enumerate(features):
        plt.arrow(0, 0, loadings_pc1[i] * scale_arrow, loadings_pc2[i] * scale_arrow,
                  color='r', alpha=0.7, head_width=0.05 * scale_arrow, head_length=0.05 * scale_arrow)
        plt.text(loadings_pc1[i] * scale_arrow * 1.1, loadings_pc2[i] * scale_arrow * 1.1,
                 feature, color='g', ha='center', va='center')

    plt.xlabel("Componente Principal 1")
    plt.ylabel("Componente Principal 2")
    plt.title("Biplot de PCA (Componentes 1 y 2)")
    plt.grid(True)
    plt.legend()

    # Save the plot
    filename = os.path.join(folder_path, f"biplot_{timestamp}.png")
    plt.savefig(filename, bbox_inches="tight")
    plt.close()
    print(f"Biplot guardado en '{filename}'.")
    return filename

def perform_anova(data, dependent_variable, grouping_variable):
    """Performs ANOVA and returns the results."""
    formula = f'{dependent_variable} ~ C({grouping_variable})'
    model = ols(formula, data=data).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    return anova_table

def perform_chi2(data, col1, col2):
    """Performs Chi-Square test of independence and returns the results."""
    contingency_table = pd.crosstab(data[col1], data[col2])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    return chi2, p, dof, expected

def describe_clusters_with_text(data_with_clusters, doc, text_cols):
    """
    Adds a description of each cluster to the Word document,
    including the most frequent values for the specified text columns.
    """
    doc.add_heading('Descripción de Clusters', level=2)

    for cluster_num in sorted(data_with_clusters['Cluster'].unique()):
        cluster_data = data_with_clusters[data_with_clusters['Cluster'] == cluster_num]
        doc.add_heading(f'Cluster {cluster_num}', level=3)

        description = f"Tamaño del cluster: {len(cluster_data)}\n"

        for col in text_cols:
            # Calculate most frequent values
            top_values = cluster_data[col].value_counts().nlargest(3).index.tolist()
            description += f"Top {col}: {', '.join(map(str, top_values))}\n"

        # Calculate the mean of numeric columns for the description
        numeric_description = cluster_data.mean(numeric_only=True).round(2).to_string()
        description += f"Valores promedio de variables numéricas:\n{numeric_description}\n"

        doc.add_paragraph(description)

def comprehensive_pca_analysis(data_numeric, data_text, timestamp, folder_path, doc):
    """Performs complete PCA analysis with rotation and evaluation."""

    # Store the original index for later combination
    original_index = data_numeric.index

    # Scale data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data_numeric)

    # KMO and Bartlett's Test
    kmo_all, kmo_model = calculate_kmo(data_numeric)
    bartlett_chi2, bartlett_p = calculate_bartlett_sphericity(data_numeric)

    # Adequacy evaluation
    kmo_evaluation = "Excelente" if kmo_model >= 0.9 else \
                     "Muy bueno" if kmo_model >= 0.8 else \
                     "Aceptable" if kmo_model >= 0.7 else \
                     "Mediocre" if kmo_model >= 0.6 else \
                     "Inaceptable"

    bartlett_evaluation = "Adecuado (p < 0.05)" if bartlett_p < 0.05 else "No adecuado (p >= 0.05)"

    # PCA without rotation
    pca_unrotated = PCA(n_components=min(data_scaled.shape[1], data_scaled.shape[0]))
    pca_unrotated.fit(data_scaled)
    pca_unrotated_scores = pca_unrotated.transform(data_scaled)

    # PCA with Varimax rotation
    fa = FactorAnalyzer(rotation='varimax', n_factors=min(data_scaled.shape[1], data_scaled.shape[0]))
    fa.fit(data_scaled)
    rotated_loadings = fa.loadings_

    # Unrotated PCA results
    unrotated_loadings = pca_unrotated.components_.T * np.sqrt(pca_unrotated.explained_variance_)
    unrotated_loadings_df = pd.DataFrame(
        unrotated_loadings,
        columns=[f"PC{i+1}" for i in range(pca_unrotated.n_components_)],
        index=data_numeric.columns
    )

    # Rotated PCA results
    rotated_loadings_df = pd.DataFrame(
        rotated_loadings,
        columns=[f"Factor{i+1}" for i in range(fa.n_factors)],
        index=data_numeric.columns
    )

    # Clustering
    optimal_k = determine_optimal_clusters(data_scaled)
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(data_scaled)

    # Create a new dataframe including the original data, PCA scores, and cluster
    data_with_clusters = data_numeric.copy()
    data_with_clusters['Cluster'] = cluster_labels

    # Add the text columns to data_with_clusters
    data_with_clusters['Litologia'] = data_text['Litologia']
    data_with_clusters['Color'] = data_text['Color']

   # Create PCA scores DataFrame
    pca_scores_df = pd.DataFrame(pca_unrotated_scores, columns=[f"PC{i+1}" for i in range(pca_unrotated.n_components_)])
    pca_scores_df['Cluster'] = cluster_labels
    pca_scores_df['Sample'] = data_numeric.index

    # Explained variance without rotation
    variance_unrotated = pd.DataFrame({
        "Componente": [f"PC{i+1}" for i in range(pca_unrotated.n_components_)],
        "Varianza Individual": pca_unrotated.explained_variance_ratio_,
        "Varianza Acumulada": np.cumsum(pca_unrotated.explained_variance_ratio_)
    })

    # Explained variance with rotation
    variance_rotated = pd.DataFrame({
        "Factor": [f"Factor{i+1}" for i in range(fa.n_factors)],
        "Varianza Individual": fa.get_factor_variance()[1],
        "Varianza Acumulada": np.cumsum(fa.get_factor_variance()[1])
    })

    # Generate plots
    var_unrotated_plot = plot_variance_explained(pca_unrotated.explained_variance_ratio_, "Varianza Explicada (PCA sin rotación)", timestamp, folder_path, "unrotated")
    var_rotated_plot = plot_variance_explained(fa.get_factor_variance()[1], "Varianza Explicada (PCA con rotación Varimax)", timestamp, folder_path, "rotated")
    circle_unrotated = plot_correlation_circle_improved(unrotated_loadings[:, :2], data_numeric.columns, pca_unrotated.explained_variance_ratio_[:2], "Círculo de Correlación (PCA sin rotación - Componentes 1-2)", timestamp, folder_path, "unrotated")
    circle_rotated = plot_correlation_circle_improved(rotated_loadings[:, :2], data_numeric.columns, fa.get_factor_variance()[1][:2], "Círculo de Correlación (PCA con rotación Varimax)", timestamp, folder_path, "unrotated")
    biplot_filename = plot_biplot(pca_unrotated_scores[:, 0], pca_unrotated_scores[:, 1], unrotated_loadings[:, 0], unrotated_loadings[:, 1], data_numeric.columns, timestamp, folder_path)
    pca = PCA(n_components=2)
    pca_scores = pca.fit_transform(data_scaled)
    clusters_plot = plot_pca_clusters(pca_scores, cluster_labels, timestamp, folder_path)

    # Call the function to describe clusters with text
    text_cols = ['Litologia', 'Color']
    describe_clusters_with_text(data_with_clusters, doc, text_cols)

    # Excel output
    excel_filename = os.path.join(folder_path, f"PCA_Results_{timestamp}.xlsx")
    with pd.ExcelWriter(excel_filename, engine='xlsxwriter') as writer:
        pca_scores_df.to_excel(writer, sheet_name='PCA Scores', index=False)
        unrotated_loadings_df.to_excel(writer, sheet_name='Unrotated Loadings', index=True)
        rotated_loadings_df.to_excel(writer, sheet_name='Rotated Loadings', index=True)
        variance_unrotated.to_excel(writer, sheet_name='Variance Unrotated', index=False)
        variance_rotated.to_excel(writer, sheet_name='Variance Rotated', index=False)
        data_with_clusters.to_excel(writer, sheet_name='Data with Clusters', index=False)

    print(f"Resultados de PCA guardados en '{excel_filename}'.")

    # Add results to Word document
    doc.add_heading('Resultados del Análisis PCA', level=1)
    doc.add_heading('Evaluación de Adecuación', level=2)
    doc.add_paragraph(f"KMO: {kmo_model:.3f} ({kmo_evaluation})")
    doc.add_paragraph(f"Bartlett: chi2={bartlett_chi2:.3f}, p={bartlett_p:.3f} ({bartlett_evaluation})")

    add_df_to_word(doc, variance_unrotated, "Varianza Explicada (PCA sin rotación)")
    add_df_to_word(doc, variance_rotated, "Varianza Explicada (PCA con rotación Varimax)")
    add_df_to_word(doc, unrotated_loadings_df, "Loadings sin rotación")
    add_df_to_word(doc, rotated_loadings_df, "Loadings con rotación (Varimax)")
    add_df_to_word(doc, data_with_clusters, "Datos con Clusters")

    # Add plots to Word document
    doc.add_heading('Gráficos', level=2)
    doc.add_paragraph("Varianza Explicada (sin rotación)")
    doc.add_picture(var_unrotated_plot, width=Inches(6))
    doc.add_paragraph("Varianza Explicada (con rotación Varimax)")
    doc.add_picture(var_rotated_plot, width=Inches(6))
    doc.add_paragraph("Círculo de Correlación (sin rotación)")
    doc.add_picture(circle_unrotated, width=Inches(6))
    doc.add_paragraph("Círculo de Correlación (con rotación Varimax)")
    doc.add_picture(circle_rotated, width=Inches(6))
    doc.add_paragraph("Biplot")
    doc.add_picture(biplot_filename, width=Inches(6))
    doc.add_paragraph("Clusters")
    doc.add_picture(clusters_plot, width=Inches(6))

    return data_with_clusters

# Example usage
if __name__ == "__main__":
    # Sample data
    data = pd.read_excel("Data.xlsx")

    # Configuration
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    folder_path = "PCA_Results_" + timestamp
    os.makedirs(folder_path, exist_ok=True)
    doc = docx.Document()

    # Separate numeric and text columns
    data_numeric = data.select_dtypes(include=[np.number])
    data_text = data.select_dtypes(exclude=[np.number])

    # Run PCA and add results to document
    data_with_clusters = comprehensive_pca_analysis(data_numeric, data_text, timestamp, folder_path, doc)

    # Save Word document
    doc_filename = os.path.join(folder_path, f"PCA_Report_{timestamp}.docx")
    doc.save(doc_filename)
    print(f"Informe guardado en '{doc_filename}'.")


/usr/local/lib/python3.11/dist-packages/factor_analyzer/utils.py:244: UserWarning: The inverse of the variance-covariance matrix was calculated using the Moore-Penrose generalized matrix inversion, due to its determinant being at or very close to zero.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Biplot guardado en 'PCA_Results_20250411_151316/biplot_20250411_151316.png'.
Resultados de PCA guardados en 'PCA_Results_20250411_151316/PCA_Results_20250411_151316.xlsx'.
Informe guardado en 'PCA_Results_20250411_151316/PCA_Report_20250411_151316.docx'.
